In [ ]:
pip install --upgrade wandb

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np, os, random, time, copy, gc
from PIL import Image
import matplotlib.pyplot as plt, seaborn as sns, cv2
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import wandb

# ===================== CONFIG =====================
# PLOS ONE: Mohamed et al. (2023) — EOSA-CNN
# "Automatic detection and classification of lung cancer CT scans
#  based on deep learning and ebola optimization search algorithm"
# Paper accuracy: 93.21% (EOSA-CNN), 80% (base CNN)
# THIS FILE: Sequential 70/20/10 split (honest — no shuffle)
# We reproduce the BASE CNN architecture (no metaheuristic)
# ==================================================

config = {
    'data_dir': '/kaggle/input/datasets/hamdallak/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset',
    'batch_size': 32, 'learning_rate': 1e-3, 'epochs': 30,
    'dropout': 0.5, 'lr_decay_factor': 0.3, 'lr_patience': 5,
    'train_ratio': 0.70, 'val_ratio': 0.20, 'test_ratio': 0.10, 'image_size': 258,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'seed': 42,
    'class_names': ['Benign', 'Malignant', 'Normal'],
    'aug_factor_benign': 13, 'aug_factor_malignant': 2, 'aug_factor_normal': 3,
}

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
set_seed(config['seed'])

CLASS_MAP = {"Bengin cases": 0, "Malignant cases": 1, "Normal cases": 2}

def collect_all_images(data_dir):
    all_samples = []
    for cls_name, cls_id in CLASS_MAP.items():
        cls_dir = os.path.join(data_dir, cls_name)
        if not os.path.exists(cls_dir):
            for split in ['train','test','valid']:
                d = os.path.join(data_dir, split, cls_name)
                if os.path.exists(d): cls_dir = d; break
        if os.path.exists(cls_dir):
            files = sorted([f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
            for f in files: all_samples.append((os.path.join(cls_dir, f), cls_id))
    return all_samples

# ===================== PLOS ONE CNN MODEL =====================
# 4 blocks: Conv(3x3,32)->Conv(3x3,32)->MaxPool(2x2)
#           Conv(3x3,64)->Conv(3x3,64)->MaxPool(3x3)
#           Conv(3x3,128)->Conv(3x3,128)->MaxPool(2x2)
#           Conv(3x3,256)->Conv(3x3,256)->MaxPool(3x3)
# Flatten -> FC(512) -> Dropout(0.5) -> Dense(3, softmax)
class PLOS_CNN(nn.Module):
    def __init__(self, num_classes=3, dropout=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(3, 3))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2))
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(256),
            nn.MaxPool2d(3, 3))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.LazyLinear(512), nn.ReLU(),
            nn.Dropout(p=dropout), nn.Linear(512, num_classes))

    def forward(self, x):
        x = self.block1(x); x = self.block2(x)
        x = self.block3(x); x = self.block4(x)
        return self.classifier(x)

# ===================== DATASET =====================
class LungDataset(Dataset):
    def __init__(self, samples, transform, oversample_factors=None):
        self.transform = transform
        if oversample_factors:
            self.samples = []
            for path, label in samples:
                self.samples.extend([(path, label)] * oversample_factors.get(label, 1))
        else: self.samples = list(samples)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

def get_transforms(sz):
    train_tf = transforms.Compose([transforms.Resize((sz,sz)),
        transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2),
        transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    eval_tf = transforms.Compose([transforms.Resize((sz,sz)),
        transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    return train_tf, eval_tf

# ===================== SEQUENTIAL SPLIT (NO SHUFFLE) =====================
def prepare_sequential_data(data_dir, train_ratio=0.70, val_ratio=0.20, batch_size=32, image_size=258):
    all_samples = collect_all_images(data_dir)
    print(f"Total images: {len(all_samples)}")
    for name, cid in CLASS_MAP.items():
        print(f"  {name}: {sum(1 for _,l in all_samples if l==cid)}")
    train_s, val_s, test_s = [], [], []
    for cls_name, cls_id in CLASS_MAP.items():
        ci = [(p,l) for p,l in all_samples if l==cls_id]; n=len(ci)
        te=int(n*train_ratio); ve=int(n*(train_ratio+val_ratio))
        train_s.extend(ci[:te]); val_s.extend(ci[te:ve]); test_s.extend(ci[ve:])
    print(f"\nAfter sequential 70/20/10 split:")
    print(f"  Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}")
    train_tf, eval_tf = get_transforms(image_size)
    aug = {0: config['aug_factor_benign'], 1: config['aug_factor_malignant'], 2: config['aug_factor_normal']}
    train_ds = LungDataset(train_s, train_tf, oversample_factors=aug)
    val_ds = LungDataset(val_s, eval_tf); test_ds = LungDataset(test_s, eval_tf)
    print(f"Train after augmentation: {len(train_ds)}")
    kw = dict(num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, **kw),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False, **kw),
            DataLoader(test_ds, batch_size=batch_size, shuffle=False, **kw))

# ===================== TRAINING =====================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train(); loss_sum, correct, total = 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), (torch.tensor(labels, device=device) if not isinstance(labels, torch.Tensor) else labels.to(device))
        optimizer.zero_grad(); out = model(imgs); loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()*imgs.size(0); correct += (out.argmax(1)==labels).sum().item(); total += imgs.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval(); loss_sum, correct, total = 0., 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), (torch.tensor(labels, device=device) if not isinstance(labels, torch.Tensor) else labels.to(device))
        out = model(imgs); loss = criterion(out, labels)
        loss_sum += loss.item()*imgs.size(0); correct += (out.argmax(1)==labels).sum().item(); total += imgs.size(0)
        all_preds.extend(out.argmax(1).cpu().tolist()); all_labels.extend(labels.cpu().tolist())
        all_probs.extend(F.softmax(out, dim=1).cpu().tolist())
    return loss_sum/total, correct/total, all_preds, all_labels, all_probs

def train_full(model, train_ld, val_ld, test_ld, device, epochs=30, lr=1e-3):
    model = model.to(device); criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
        factor=config['lr_decay_factor'], patience=config['lr_patience'])
    best_val_acc, best_wts, no_improve, patience = 0, None, 0, 7
    print(f"\n{'='*60}\n Training PLOS-CNN | {epochs} epochs | LR={lr}\n{'='*60}")
    for ep in range(epochs):
        t0 = time.time()
        tr_l, tr_a = train_one_epoch(model, train_ld, optimizer, criterion, device)
        vl_l, vl_a, vl_p, vl_lb, _ = evaluate(model, val_ld, criterion, device)
        vl_f1 = f1_score(vl_lb, vl_p, average='macro', zero_division=0)
        scheduler.step(vl_a); cur_lr = optimizer.param_groups[0]['lr']
        mk = ''
        if vl_a > best_val_acc: best_val_acc=vl_a; best_wts=copy.deepcopy(model.state_dict()); no_improve=0; mk=' ★'
        else: no_improve += 1
        print(f"  [{ep+1:3d}/{epochs}] loss={tr_l:.4f}/{vl_l:.4f} acc={tr_a:.4f}/{vl_a:.4f} f1={vl_f1:.4f} lr={cur_lr:.1e} ({time.time()-t0:.1f}s){mk}")
        wandb.log({"epoch":ep+1,"train/loss":tr_l,"train/acc":tr_a*100,"val/loss":vl_l,"val/acc":vl_a*100,"val/f1":vl_f1*100,"best_val_acc":best_val_acc*100,"learning_rate":cur_lr})
        if no_improve >= patience: print(f"  ⏹ Early stop at epoch {ep+1}"); break
    if best_wts: model.load_state_dict(best_wts)
    _, test_acc, test_p, test_l, test_pr = evaluate(model, test_ld, criterion, device)
    return model, test_acc, test_p, test_l, test_pr

# ===================== XAI: Grad-CAM =====================
def gradcam(model, test_ld, device, class_names):
    model.eval(); target = model.block4[-3]  # last Conv2d in block4
    samples = {}
    for imgs, labels in test_ld:
        for i in range(imgs.size(0)):
            lbl = labels[i].item() if isinstance(labels[i], torch.Tensor) else labels[i]
            if lbl not in samples: samples[lbl] = imgs[i]
        if len(samples) >= 3: break
    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    for row, (lbl, img_t) in enumerate(sorted(samples.items())):
        img_np = img_t.permute(1,2,0).cpu().numpy()
        img_np = img_np * np.array([.229,.224,.225]) + np.array([.485,.456,.406])
        img_np = np.clip(img_np, 0, 1)
        acts, grads = {}, {}
        h1 = target.register_forward_hook(lambda m,i,o: acts.update({'v':o.detach()}))
        h2 = target.register_full_backward_hook(lambda m,gi,go: grads.update({'v':go[0].detach()}))
        inp = img_t.unsqueeze(0).to(device).requires_grad_(True)
        out = model(inp); model.zero_grad(); out[0,lbl].backward()
        h1.remove(); h2.remove()
        w = grads['v'][0].mean(dim=(1,2), keepdim=True)
        cam = (w * acts['v'][0]).sum(0).cpu().numpy(); cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (config['image_size'], config['image_size']))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        axes[row,0].imshow(img_np); axes[row,0].set_title(f'Original\nTrue: {class_names[lbl]}'); axes[row,0].axis('off')
        axes[row,1].imshow(cam, cmap='jet'); axes[row,1].set_title('Grad-CAM'); axes[row,1].axis('off')
        overlay = np.clip(0.5*img_np + 0.5*plt.cm.jet(cam)[:,:,:3], 0, 1)
        axes[row,2].imshow(overlay); axes[row,2].set_title('Overlay'); axes[row,2].axis('off')
    fig.suptitle('XAI: Grad-CAM (PLOS-CNN Block4 Last Conv)', fontsize=15, fontweight='bold')
    plt.tight_layout(); wandb.log({"xai/gradcam": wandb.Image(fig)}); plt.show(); plt.close()
    print("\n✅ Grad-CAM visualizations generated.")

# ===================== MAIN =====================
if __name__ == '__main__':
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("wandb_api_key"))
    wandb.init(project="plos_one_lung_reproduction", name="plos_cnn_sequential_split", config=config,
        tags=["sequential-split","honest","70-20-10","plos-one"], notes="PLOS ONE CNN: 70/20/10 sequential split (honest)")
    print("="*60); print("PLOS ONE CNN — Sequential 70/20/10 Split (Honest)"); print("="*60)
    train_ld, val_ld, test_ld = prepare_sequential_data(config['data_dir'],
        train_ratio=config['train_ratio'], val_ratio=config['val_ratio'],
        batch_size=config['batch_size'], image_size=config['image_size'])
    wandb.log({"dataset/train_samples":len(train_ld.dataset),"dataset/val_samples":len(val_ld.dataset),
               "dataset/test_samples":len(test_ld.dataset),"dataset/split_method":"sequential 70/20/10 (no shuffle)"})

    model = PLOS_CNN(num_classes=3, dropout=config['dropout'])
    # Dry run to init LazyLinear
    with torch.no_grad(): model(torch.randn(1, 3, config['image_size'], config['image_size']))
    total_p = sum(p.numel() for p in model.parameters())
    print(f"  PLOS-CNN: {total_p/1e6:.2f}M params")

    model, test_acc, test_preds, test_labels, test_probs = train_full(
        model, train_ld, val_ld, test_ld, config['device'], config['epochs'], config['learning_rate'])
    print(f"\n{'='*60}\n TEST RESULTS — PLOS-CNN (Honest)\n{'='*60}")
    print(f"Accuracy: {test_acc*100:.2f}%\n")
    print(classification_report(test_labels, test_preds, target_names=config['class_names'], zero_division=0))
    cm = confusion_matrix(test_labels, test_preds)
    fig, ax = plt.subplots(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=config['class_names'],
        yticklabels=config['class_names'], ax=ax, annot_kws={'size':16})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('PLOS-CNN Confusion Matrix')
    plt.tight_layout(); wandb.log({"confusion_matrix": wandb.Image(fig)}); plt.show(); plt.close()
    try: auc = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='macro')
    except: auc = 0
    wandb.log({"test/acc":test_acc*100,"test/auc":auc*100})
    print(f"\nGenerating XAI..."); gradcam(model, test_ld, config['device'], config['class_names'])
    wandb.finish(); print("Done!")
